# Clase 212 — DuckDB (full demo) + snippets BQ/Snowflake

DuckDB se corre local sin credenciales. BQ y Snowflake requieren cuenta — los mostramos como código de referencia.

In [ ]:
import duckdb, pandas as pd, tempfile, time
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'dw_demo'
WORK.mkdir(exist_ok=True)
print('DuckDB version:', duckdb.__version__)

## 1. DuckDB: tabla particionada por fecha

In [ ]:
con = duckdb.connect(str(WORK / 'warehouse.duckdb'))

# Generar 2M rows × 90 días = 180M synthetic
con.execute('''
    CREATE OR REPLACE TABLE trips AS
    SELECT
        i AS trip_id,
        (random() * 100)::INT AS zone_id,
        random() * 100 + 5 AS fare,
        random() * 20 AS tip,
        CAST('2024-01-01' AS DATE) + INTERVAL ((random() * 90)::INT) DAY AS pickup_date,
        CASE WHEN random() < 0.25 THEN 'Manhattan'
             WHEN random() < 0.5 THEN 'Brooklyn'
             WHEN random() < 0.75 THEN 'Queens' ELSE 'Bronx' END AS borough
    FROM range(2_000_000) t(i)
''')
print(con.execute('SELECT COUNT(*), MIN(pickup_date), MAX(pickup_date) FROM trips').fetchone())

In [ ]:
# Exportar particionado por pickup_date (Hive-style partitioning)
out = WORK / 'trips_partitioned'
con.execute(f'''
    COPY (SELECT * FROM trips) TO '{out}' (FORMAT PARQUET, PARTITION_BY (pickup_date))
''')

subdirs = sorted([p.name for p in out.iterdir() if p.is_dir()])[:5]
print('subdirs:', subdirs)

# Predicate pushdown al leer
t0 = time.perf_counter()
result = con.execute(f"SELECT borough, AVG(fare) FROM read_parquet('{out}/**/*.parquet', hive_partitioning=true) WHERE pickup_date = '2024-01-15' GROUP BY borough").fetchdf()
print(f'query filtrada 1 día: {(time.perf_counter() - t0) * 1000:.1f} ms')
print(result)

## 2. DuckDB queryando S3 directo (sin descargar)

In [ ]:
remote_snippet = '''\
-- DuckDB lee Parquet remoto con HTTPFS extension
INSTALL httpfs; LOAD httpfs;

-- Para S3 público:
SELECT borough, COUNT(*) FROM read_parquet('s3://bucket/path/*.parquet') GROUP BY borough LIMIT 10;

-- Con credenciales:
SET s3_access_key_id = '...';
SET s3_secret_access_key = '...';
SET s3_region = 'us-east-1';
'''
print(remote_snippet)

## 3. BigQuery snippets

In [ ]:
bq_snippet = '''\
from google.cloud import bigquery
client = bigquery.Client(project="my-project")

# 1) Query con cost guard
job_config = bigquery.QueryJobConfig(maximum_bytes_billed=10 * 1024**3)  # 10 GB hard cap
q = """
  SELECT pickup_borough, COUNT(*) AS trips, AVG(fare_amount) AS avg_fare
  FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2018`
  WHERE DATE(pickup_datetime) BETWEEN "2018-01-01" AND "2018-01-31"
  GROUP BY pickup_borough
"""
df = client.query(q, job_config=job_config).to_dataframe()

# 2) Dry-run para estimar bytes ANTES de pagar
dry = bigquery.QueryJobConfig(dry_run=True)
job = client.query(q, job_config=dry)
print(f"Estimado: {job.total_bytes_processed / 1024**3:.2f} GB")

# 3) CREATE TABLE particionada + clusterizada
client.query("""
  CREATE OR REPLACE TABLE `my.dataset.trips_part`
  PARTITION BY DATE(pickup_datetime)
  CLUSTER BY pickup_borough, zone_id AS
  SELECT * FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2018`
""").result()
'''
print(bq_snippet)

## 4. Snowflake snippets

In [ ]:
sf_snippet = '''\
import snowflake.connector
ctx = snowflake.connector.connect(
    user="USER", password="...", account="xy12345.us-east-1",
    warehouse="COMPUTE_WH", database="MY_DB", schema="PUBLIC",
)
cur = ctx.cursor()

# 1) Crear tabla con cluster keys
cur.execute("""
  CREATE OR REPLACE TABLE trips (
    trip_id INT, zone_id INT, fare FLOAT, pickup_date DATE, borough STRING
  ) CLUSTER BY (pickup_date, borough)
""")

# 2) Bulk ingest desde S3 con COPY INTO
cur.execute("""
  COPY INTO trips FROM 's3://bucket/trips/'
  CREDENTIALS = (AWS_KEY_ID='...' AWS_SECRET_KEY='...')
  FILE_FORMAT = (TYPE = PARQUET)
""")

# 3) Time travel — recuperar después de un DELETE accidental
cur.execute("DELETE FROM trips WHERE borough = 'Bronx'")
# Oops! Recuperar:
cur.execute("""
  CREATE TABLE trips_restored AS
  SELECT * FROM trips AT(OFFSET => -60)   -- 60 seg atrás
""")

# 4) Snowflake escala compute on-demand
cur.execute("USE WAREHOUSE COMPUTE_L_WH")   # cambiar a warehouse más grande
'''
print(sf_snippet)

## 5. Comparativa de decisión

In [ ]:
import pandas as pd
decision = pd.DataFrame([
    {'aspect': 'Setup',            'DuckDB': 'pip install', 'BigQuery': 'cuenta GCP + SA',     'Snowflake': 'trial 30d'},
    {'aspect': 'Costo idle',       'DuckDB': '$0',           'BigQuery': '$0',                  'Snowflake': '$0 (con auto-suspend)'},
    {'aspect': 'Costo por query',  'DuckDB': '$0',           'BigQuery': '$5/TB scanned',       'Snowflake': '$/hr de VW'},
    {'aspect': 'Escala',           'DuckDB': '1 máquina',    'BigQuery': 'PB elástico',         'Snowflake': 'PB elástico'},
    {'aspect': 'Concurrencia',     'DuckDB': 'lectores OK, 1 escritor', 'BigQuery': 'alta',     'Snowflake': 'alta'},
    {'aspect': 'Time travel',      'DuckDB': 'no',           'BigQuery': 'snapshots',           'Snowflake': '90 días built-in'},
    {'aspect': 'Ideal cuando',     'DuckDB': 'local/dev/análisis', 'BigQuery': 'GCP, serverless', 'Snowflake': 'multi-cloud + sharing'},
])
print(decision.to_string(index=False))

In [ ]:
con.close()

## Ejercicio guiado

1. Migrá un script pandas que procesa CSVs grandes a DuckDB con SQL. Compará tiempo y RAM.
2. Si tenés acceso BQ: hacé `dry_run` antes de cada query. Documentá ahorro.
3. Activá trial de Snowflake. Cargá un parquet con `COPY INTO`. Hacé `DELETE` accidental y recuperá con time travel.
4. Diseñá una tabla particionada + clusterizada para tu dominio. Justificá las elecciones.
5. Calculá costo de tu workload típico en cada uno de los 3 DW.

## Conclusiones

- DuckDB es "el data warehouse que entra en `pip install`" — usalo siempre que entre en una máquina.
- BigQuery: el más simple para arrancar serverless; pero `SELECT *` sin partition filter te puede arruinar.
- Snowflake: el más feature-rich (time travel, sharing) y multi-cloud; cobra por compute time.
- Particionado + clustering son la diferencia entre query de 5 s y de 500 s.